# UFC Fight Prediction — Machine Learning

**Goal:** predict the winner of a UFC fight from each fighter's historical and profile features.

**Data:** features engineered in a dbt + DuckDB pipeline (`ml_fight_dataset`), one row per fight
with both fighters' stats side by side and the target `a_won` (did fighter A win).

**Methodology (key points):**
- All features are computed from **past fights only** (no data leakage).
- Train/test split is done **by time** (train on old fights, test on recent ones) to mimic real prediction.
- Features are **standardized** (fit on train only) before logistic regression.
- Model performance is always compared to a **baseline** (predicting the majority class).

**Result:** a logistic regression reaches **~0.60 accuracy** on unseen recent fights
(baseline ~0.51), which is solid for MMA, where outcomes are inherently hard to predict.


## 1. Load the data

We read the ML-ready table from DuckDB in **read-only** mode (so it doesn't conflict with
other tools). Each row is one fight with both fighters' features and the target `a_won`.

In [1]:
import duckdb
import pandas as pd

# Read-only connection avoids the single-writer lock of DuckDB
con = duckdb.connect("data/ufc.duckdb", read_only=True)
df = con.sql("SELECT * FROM ml_fight_dataset").df()
con.close()

print("Shape:", df.shape)
df.head()

Shape: (8701, 27)


,fight_id,event_date,fighter_a,a_fights_before,a_wins_before,a_win_rate,a_ko_rate,a_sub_rate,a_won_previous,a_days_since_last,...,b_win_rate,b_ko_rate,b_sub_rate,b_won_previous,b_days_since_last,b_age,b_reach,b_stance,b_weight_change,a_won
0,f5686732b85adafc,2010-12-04,Aaron Wilkinson,0,0.0,0.0,0.0,0.0,<NA>,<NA>,...,0.0,0.0,0.0,<NA>,<NA>,23,72,Switch,<NA>,0
1,b22dbe21c21060b9,2015-12-11,Abner Lloveras,0,0.0,0.0,0.0,0.0,<NA>,<NA>,...,0.0,0.0,0.0,<NA>,<NA>,29,68,Orthodox,<NA>,0
2,1c245dc63a0b9d51,2012-06-01,Al Iaquinta,0,0.0,0.0,0.0,0.0,<NA>,<NA>,...,0.0,0.0,0.0,<NA>,<NA>,25,75,Southpaw,<NA>,0
3,9cd973a0a6c1cd5c,2013-08-31,Al Iaquinta,1,0.0,0.0,0.0,0.0,0,456,...,0.0,0.0,0.0,0,147,31,73,Orthodox,0,1
4,22fdf7a08691db5d,2013-10-26,Al Iaquinta,2,1.0,0.5,0.0,0.0,1,56,...,1.0,0.0,1.0,1,52,26,71,Orthodox,0,1


## 2. Explore the dataset

We check the columns, the target balance, and define the **baseline** to beat.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8701 entries, 0 to 8700
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   fight_id           8701 non-null   str           
 1   event_date         8701 non-null   datetime64[us]
 2   fighter_a          8701 non-null   str           
 3   a_fights_before    8701 non-null   int64         
 4   a_wins_before      8701 non-null   float64       
 5   a_win_rate         8701 non-null   float64       
 6   a_ko_rate          8701 non-null   float64       
 7   a_sub_rate         8701 non-null   float64       
 8   a_won_previous     7347 non-null   Int32         
 9   a_days_since_last  7347 non-null   Int64         
 10  a_age              8573 non-null   Int64         
 11  a_reach            7996 non-null   Int32         
 12  a_stance           8634 non-null   str           
 13  a_weight_change    7137 non-null   Int32         
 14  fighter_b          

**Target balance** — `a_won` is the label we predict (1 if fighter A won).
Since A/B is assigned alphabetically (arbitrary), the target is ~50/50, which is healthy.

In [3]:
# Target distribution -> defines the baseline (predicting the majority class)
print(df["a_won"].value_counts(normalize=True))
print("\nBaseline (always predict majority class): "
      f"{df['a_won'].value_counts(normalize=True).max():.3f}")

a_won
0    0.510631
1    0.489369
Name: proportion, dtype: float64

Baseline (always predict majority class): 0.511


## 3. Handle missing values

Missing values come from two sources:
- **First fights**: no previous fight yet (no form / inactivity info).
- **Missing profiles**: some fighters have no reach / age / stance in the source data.

We fill them with sensible defaults:
- numeric profile features (age, reach) -> **median** (a typical value, avoids absurd zeros)
- form / inactivity for debuts -> neutral defaults
- `weight_change` -> 0 (no known change)

*Note: `stance` (text) is kept as-is here; it was tested but dropped from the final model
because it carried almost no predictive signal.*

In [4]:
# Numeric profile features -> median
for col in ["a_age", "b_age", "a_reach", "b_reach"]:
    df[col] = df[col].fillna(df[col].median())

# First-fight defaults
df["a_won_previous"]    = df["a_won_previous"].fillna(0)
df["b_won_previous"]    = df["b_won_previous"].fillna(0)
df["a_days_since_last"] = df["a_days_since_last"].fillna(9999)
df["b_days_since_last"] = df["b_days_since_last"].fillna(9999)

# Weight-class change: NULL -> 0 (no known change)
df["a_weight_change"] = df["a_weight_change"].fillna(0)
df["b_weight_change"] = df["b_weight_change"].fillna(0)

# Remaining NA are only in unused text columns (stance) -> harmless
print("Missing values in used columns:",
      df.drop(columns=["a_stance", "b_stance"]).isnull().sum().sum())

Missing values in used columns: 0


## 4. Feature engineering — difference features

For each symmetric feature we compute the **difference** `A - B`.
This directly captures *who has the advantage*, which is a stronger and more robust
signal than the two absolute values (and it is independent of the arbitrary A/B order).

In [5]:
df["diff_fights_before"]   = df["a_fights_before"]   - df["b_fights_before"]
df["diff_wins_before"]     = df["a_wins_before"]     - df["b_wins_before"]
df["diff_win_rate"]        = df["a_win_rate"]        - df["b_win_rate"]
df["diff_age"]             = df["a_age"]             - df["b_age"]
df["diff_reach"]           = df["a_reach"]           - df["b_reach"]
df["diff_days_since_last"] = df["a_days_since_last"] - df["b_days_since_last"]

# weight_change difference is engineered too, but NOT used in the final model:
# only ~7.5% of fights are real category changes, so it adds no measurable signal.
df["diff_weight_change"]   = df["a_weight_change"]   - df["b_weight_change"]

df[["diff_wins_before", "diff_win_rate", "diff_age", "diff_reach"]].describe()

,diff_wins_before,diff_win_rate,diff_age,diff_reach
count,8701.000000,8701.000000,8701.0,8701.0
mean,0.044822,-0.003548,0.150213,0.044018
std,4.097105,0.385022,5.281639,3.278319
min,-24.000000,-1.000000,-25.0,-13.0
25%,-2.000000,-0.200000,-3.0,-2.0
50%,0.000000,0.000000,0.0,0.0
75%,2.000000,0.200000,4.0,2.0
max,24.000000,1.000000,23.0,12.0


## 5. Train/test split — by time (no data leakage)

The data is chronological, so we train on the **oldest 80%** of fights and test on the
**most recent 20%**. This mimics reality: we predict the future from the past.
A random split would leak future information into training and inflate the score.

In [6]:
df_sorted = df.sort_values("event_date").reset_index(drop=True)

# Final feature set: the cleaned, informative features
# (style rates, stances and weight_change were tested but dropped: no measurable gain)
feature_cols = [
    "a_fights_before", "a_wins_before", "a_win_rate",
    "b_fights_before", "b_wins_before", "b_win_rate",
    "a_days_since_last", "b_days_since_last",
    "a_age", "b_age",
    "diff_fights_before", "diff_wins_before", "diff_win_rate",
    "diff_age", "diff_reach", "diff_days_since_last",
]

X = df_sorted[feature_cols]
y = df_sorted["a_won"]

split_index = int(len(df_sorted) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train period:", df_sorted["event_date"].iloc[0].date(),
      "->", df_sorted["event_date"].iloc[split_index - 1].date())
print("Test period: ", df_sorted["event_date"].iloc[split_index].date(),
      "->", df_sorted["event_date"].iloc[-1].date())

Train: (6960, 16) | Test: (1741, 16)
Train period: 1994-03-11 -> 2023-01-21
Test period:  2023-01-21 -> 2026-05-16


## 6. Train the model — Logistic Regression

We standardize the features (**fit on train only**, to avoid leakage) and train a
logistic regression: a simple, interpretable model that works well when the signal is
mostly linear. We also tried Random Forest and XGBoost, but they overfit and did not
beat logistic regression on the test set — a reminder that, when the signal is limited,
a simpler model generalizes better.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn stats on train only
X_test_scaled  = scaler.transform(X_test)        # apply same stats to test

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

print("Model trained on", X_train.shape[0], "fights.")

Model trained on 6960 fights.


## 7. Evaluate

We measure accuracy on the **test set** (recent fights never seen in training) and compare
it to the baseline. We also inspect the confusion matrix and the feature weights.

In [8]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
baseline = y_test.value_counts(normalize=True).max()

print(f"Model accuracy: {accuracy:.3f}")
print(f"Baseline:       {baseline:.3f}")
print(f"Improvement:    {accuracy - baseline:+.3f}")

Model accuracy: 0.597
Baseline:       0.515
Improvement:    +0.082


**Confusion matrix** — shows *where* the model makes mistakes, not just the global score.

In [9]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print("                 Pred A loses   Pred A wins")
print(f"Real A loses        {cm[0][0]:>6}        {cm[0][1]:>6}")
print(f"Real A wins         {cm[1][0]:>6}        {cm[1][1]:>6}")

Confusion matrix:
                 Pred A loses   Pred A wins
Real A loses           561           335
Real A wins            367           478


**Feature importance** — standardized weights, so they are directly comparable.
Positive = pushes toward "A wins", negative = toward "A loses". Larger absolute value =
more influential. The fight record (wins / experience differences) and age dominate.

In [10]:
coefficients = pd.DataFrame({
    "feature": feature_cols,
    "weight": model.coef_[0]
})
coefficients["abs_weight"] = coefficients["weight"].abs()
coefficients.sort_values("abs_weight", ascending=False)[["feature", "weight"]]

,feature,weight
11,diff_wins_before,0.360299
10,diff_fights_before,-0.281208
4,b_wins_before,-0.236491
3,b_fights_before,0.211850
13,diff_age,-0.166505
1,a_wins_before,0.138353
8,a_age,-0.117207
0,a_fights_before,-0.100613
9,b_age,0.094084
14,diff_reach,0.086214


## 8. Predict a fight — who wins between X and Y?

To predict a matchup between two named fighters, we load each fighter's **most recent**
feature row (a proxy for their current state), build the model features for the matchup,
apply the same scaler, and output the predicted winner with a probability.

*Note: features are taken as of each fighter's last fight, so this is an approximation
of their current form — suitable for a demo.*

In [11]:
# Load each fighter's most recent feature row
con = duckdb.connect("data/ufc.duckdb", read_only=True)
latest = con.sql('''
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY fighter ORDER BY event_date DESC) AS rn
        FROM fct_fighter_features
    ) WHERE rn = 1
''').df()
con.close()

print("Fighters available:", latest.shape[0])

Fighters available: 2695


In [23]:
def predict_fight(name_a, name_b):
    fa = latest[latest["fighter"] == name_a]
    fb = latest[latest["fighter"] == name_b]
    if fa.empty: return f"Fighter not found: {name_a}"
    if fb.empty: return f"Fighter not found: {name_b}"
    fa, fb = fa.iloc[0], fb.iloc[0]

    reach_a = fa["reach"] if pd.notna(fa["reach"]) else 71
    reach_b = fb["reach"] if pd.notna(fb["reach"]) else 71

    row = {
        "a_fights_before": fa["fights_before"], "a_wins_before": fa["wins_before"], "a_win_rate": fa["win_rate"],
        "b_fights_before": fb["fights_before"], "b_wins_before": fb["wins_before"], "b_win_rate": fb["win_rate"],
        "a_days_since_last": fa["days_since_last_fight"], "b_days_since_last": fb["days_since_last_fight"],
        "a_age": fa["age_at_fight"], "b_age": fb["age_at_fight"],
        "diff_fights_before": fa["fights_before"] - fb["fights_before"],
        "diff_wins_before":   fa["wins_before"]   - fb["wins_before"],
        "diff_win_rate":      fa["win_rate"]      - fb["win_rate"],
        "diff_age":           fa["age_at_fight"]  - fb["age_at_fight"],
        "diff_reach":         reach_a - reach_b,
        "diff_days_since_last": fa["days_since_last_fight"] - fb["days_since_last_fight"],
    }
    X_row = pd.DataFrame([row])[feature_cols].fillna(0)
    proba_a = model.predict_proba(scaler.transform(X_row))[0][1]

    winner = name_a if proba_a >= 0.5 else name_b
    win_proba = proba_a if proba_a >= 0.5 else 1 - proba_a
    return f"{winner} wins (probability: {win_proba:.1%})"



# UFC Freedom 250 (White House) - card predictions
white_house_card = [
    ("Ilia Topuria", "Justin Gaethje"),
    ("Alex Pereira", "Ciryl Gane"),
    ("Sean O'Malley", "Aiemann Zahabi"),
    ("Derrick Lewis", "Josh Hokit"),
    ("Mauricio Ruffy", "Michael Chandler"),
    ("Bo Nickal", "Kyle Daukaus"),
    ("Diego Lopes", "Steve Garcia"),
]

print("UFC FREEDOM 250 - WHITE HOUSE - Model predictions\n")
for a, b in white_house_card:
    print(f"{a} vs {b}")
    print("  ->", predict_fight(a, b), "\n")

Jon Jones wins (probability: 67.6%)
Dustin Poirier wins (probability: 53.2%)
UFC FREEDOM 250 - WHITE HOUSE - Model predictions

Ilia Topuria vs Justin Gaethje
  -> Ilia Topuria wins (probability: 72.6%) 

Alex Pereira vs Ciryl Gane
  -> Ciryl Gane wins (probability: 58.0%) 

Sean O'Malley vs Aiemann Zahabi
  -> Sean O'Malley wins (probability: 61.7%) 

Derrick Lewis vs Josh Hokit
  -> Josh Hokit wins (probability: 52.6%) 

Mauricio Ruffy vs Michael Chandler
  -> Mauricio Ruffy wins (probability: 73.5%) 

Bo Nickal vs Kyle Daukaus
  -> Bo Nickal wins (probability: 66.3%) 

Diego Lopes vs Steve Garcia
  -> Steve Garcia wins (probability: 52.6%) 



## Conclusion & limitations

**What works:** the fight record (wins, experience, win rate) and the age difference are
the strongest predictors. The model reaches ~0.60 accuracy on unseen recent fights
(baseline ~0.51), a solid result for MMA.

**What we tested and dropped (measured, not assumed):**
- *Style* (KO rate / submission rate): great domain intuition, but almost no predictive gain.
- *Stance* (one-hot): negligible signal.
- *Weight-class change* (up/down/same): engineered in the pipeline, but only ~7.5% of fights
  are real changes, so it added no measurable accuracy.
- *Random Forest / XGBoost*: overfit, did not beat logistic regression on the test set.

**Limitations:**
- Features use each fighter's last-known state (approximation of current form).
- No fight-day information (injuries, camp, short-notice), no detailed striking/grappling stats.
- ~0.60 is close to the realistic ceiling with historical features alone.

**Possible next steps:** parse detailed fight statistics (`fight_stats`: strikes, takedowns,
control time) for richer style signals; add an interactive dashboard (Streamlit).
